In [1]:
#importe das bibliotecas utilizadas

import cv2
import numpy as np
import matplotlib.pyplot as plt
import time


In [2]:
#Função de tratamento de imagem em matriz

def carregar_video_como_matriz(caminho_video, largura_alvo=100, altura_alvo=100):
    cap = cv2.VideoCapture(caminho_video)
    frames = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # Converter para tons de cinza
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        # Redimensionar para reduzir custo computacional
        resized = cv2.resize(gray, (largura_alvo, altura_alvo))
        # Vetorizar a imagem (transformar matriz 2D em vetor 1D) e normalizar (opcional, 0 a 1)
        v = resized.flatten().astype(float) / 255.0
        frames.append(v)
        
    cap.release()
    # M terá dimensões (m x n), onde m = pixels e n = número de quadros
    M = np.column_stack(frames)
    return M, altura_alvo, largura_alvo

# Exemplo de uso:
# M, altura, largura = carregar_video_como_matriz("seu_video.mp4")

In [3]:
carregar_video_como_matriz("data/teste.mp4", 10, 10)

(array([[0.31764706, 0.31764706, 0.31372549, ..., 0.31372549, 0.31372549,
         0.30980392],
        [0.16078431, 0.15686275, 0.16078431, ..., 0.29803922, 0.29411765,
         0.29411765],
        [0.17254902, 0.17647059, 0.17647059, ..., 0.27058824, 0.27058824,
         0.27058824],
        ...,
        [0.4745098 , 0.49019608, 0.46666667, ..., 0.22352941, 0.22745098,
         0.23137255],
        [0.36078431, 0.40784314, 0.38039216, ..., 0.34117647, 0.35294118,
         0.34117647],
        [0.34901961, 0.34509804, 0.35686275, ..., 0.34901961, 0.34509804,
         0.34901961]], shape=(100, 330)),
 10,
 10)

In [2]:
#Funções de decomposição SVD

#Definicao da matriz devidamente formatada
def Matriz(A):
    M = A@(np.transpose(A))
    return M

#Decompoe as matrizes Q e R
def Matrizes_Q_R(M):
    Q, R = np.linalg.qr(M)
    return Q, R

#Calculo dos valores singulares
def Singulares_QR(B):

   B = np.array(B)
   tol = 1e-15
   W = B.copy()

   for k in range(1000):
      Qk, Rk = Matrizes_Q_R(W)
      Wr = W.copy()
      W = Rk @ Qk
      er = np.linalg.norm(np.diag(W) - np.diag(Wr)) / np.linalg.norm(np.diag(W)) 
      er = np.abs(er)

      if er < tol:
         break

   autoval = []
   for j in range(W.shape[0]):
      autoval.append(np.max(np.sqrt(W[j][j].real),0))

   autoval.sort(reverse= True)
   autoval = np.array(autoval)
   return autoval

         
def Autovetores_Direita(A, autv):
   tol=1e-15
   iter= 1000
   A = np.array(A)
   n= A.shape[0]
   I = np.eye(n)
   autovetores = []

   e = 1e-15

   
   for k in range(n):
      qold = np.random.rand(n)
      mu = autv[k] + e
      B= (A - mu*I)

      norm = 1                                       
      cont = 0                                       
      while norm > tol and cont < iter:              
         qnew = np.linalg.solve(B,qold)              
         qnew = qnew/np.linalg.norm(qnew,ord=2)      
         norm = 1 - np.abs(np.vdot(qold, qnew))      
         qold = qnew                                 
         cont += 1

         autovetores.append(qold)     

   
   autovetores = np.array(autovetores)

   return autovetores

def Autovetores_esquerda(A, autovec, autoval):
   u = []
   n = len(autoval)
   for i in range(n):
      u.append((A @ autovec[i])/autoval[i])

   u = np.array(u)
   return u

def Matrizes(A, AutovecL, AutovecR, Autoval):
   n = A.shape[0]

   U = []
   S = np.eye(n)
   V = []

   for j in range(n):
        U.append(AutovecL[j])
        S[j][j] = Autoval[j]
        V.append(AutovecR[j])

   U = np.array(U)
   V = np.array(V).T

   return U, S, V

def DecompsicaoSVD(A):
   inicio = time.perf_counter()
   M = Matriz(A)
   ValoresSingulares = Singulares_QR(M)
   VecR = Autovetores_Direita(M, ValoresSingulares)
   VecL = Autovetores_esquerda(M,VecR, ValoresSingulares)
   U, S, V = Matrizes(M, VecL, VecR, ValoresSingulares)
   fim = time.perf_counter()
   tempo = fim - inicio
   return U, S, V, tempo

In [5]:
# Comparacao com o svd nativo:

A = np.array([[1,2,3],[9,0,7],[2,5,0]])

u,s,v,t= DecompsicaoSVD(A)
print(f"Minha solução eh: U: {u}\n S: {s}\n V: {v}")
print("")
print("O tempo de execucao foi de:", t)
print("")

iniProg = time.perf_counter()
m, n, p= np.linalg.svd(A)
fimProg = time.perf_counter()
tempoProg= fimProg - iniProg

print(f"A solucao do programa foi: U: {m}\n S: {n}\n V: {p}")
print("")
print("O tempo de execucao do programa foi:", tempoProg)
print("")
print(f"A diferenca foi: U: {u - m}\n S: {s - (n @ np.eye(3))}\n Vt: {v - p}")

Minha solução eh: U: [[-0.04385631  0.80319693  0.51676456]
 [ 0.84519929 -0.10235284  0.04234205]
 [-1.79364216  0.28491266  1.00653965]]
 S: [[11.87441822  0.          0.        ]
 [ 0.          5.274628    0.        ]
 [ 0.          0.          2.04364658]]
 V: [[-0.87225914  0.94530793 -0.92227213]
 [ 0.21375272 -0.18511544  0.16918901]
 [ 0.43985653 -0.26856133  0.3475474 ]]

O tempo de execucao foi de: 0.0068053719999170426

A solucao do programa foi: U: [[-0.24217417 -0.27582641  0.93019969]
 [-0.95354174  0.24473231 -0.17568222]
 [-0.17919212 -0.92952993 -0.32227984]]
 S: [11.87441822  5.274628    2.04364658]
 V: [[-0.77329549 -0.11624224 -0.62329914]
 [ 0.01283588 -0.98571927  0.16790699]
 [-0.63391586  0.12184113  0.76374434]]

O tempo de execucao do programa foi: 0.0008924789999582572

A diferenca foi: U: [[ 0.19831786  1.07902333 -0.41343513]
 [ 1.79874104 -0.34708514  0.21802427]
 [-1.61445004  1.21444259  1.32881948]]
 S: [[ 0.00000000e+00 -5.27462800e+00 -2.04364658e+00]